# MIMIC to MIMIC-on-FHIR DST-gap investigation

This notebook compares the original `mimiciv_hosp.admissions.admittime`
with the corresponding hospital Encounter `period.start` in the generated
MIMIC-on-FHIR Delta warehouse.

The comparison deliberately preserves the wall-clock digits from the FHIR
string. It does **not** parse the FHIR offset as a real instant, because that
would introduce the separate 13-16 hour local-session-timezone bug.

The goal is to expose the smaller upstream DST-gap transformation:
a source value such as `02:10` can be served by FHIR as `03:10` when the
upstream ETL casts the naive source timestamp through `TIMESTAMPTZ`.

## Interpretation guide

On a spring-forward transition, local clocks jump from `01:59:59` to
`03:00:00`. The local interval `02:00`-`02:59` therefore does not exist.
If the ETL is forced to interpret a naive MIMIC wall-clock value of
`02:10` as a timezone-aware timestamp, the timezone library normalizes it
to `03:10`. The downstream FHIR value then contains `03:10`; the original
`02:10` is no longer available to a concept mapping.

A zero-row result in the demo does not disprove the issue. The demo is a
small patient subset and may simply contain no admission in the DST gap.

In [1]:
from html import escape
import os
from pathlib import Path

import duckdb
from IPython.display import HTML, Markdown, display

# Override these with environment variables when investigating another
# warehouse. The defaults are the local demo oracle and Delta warehouse.
ORACLE_PATH = Path(os.environ.get(
    'MIMIC_DUCKDB_PATH',
    '~/warehouses/mimic4-demo.db',
))
FHIR_WAREHOUSE = Path(os.environ.get(
    'MIMIC_FHIR_WAREHOUSE',
    '~/warehouses/mimic-iv-demo/delta',
))
ENCOUNTER_GLOB = str(FHIR_WAREHOUSE / 'Encounter.parquet' / '**' / '*.parquet')
HOSPITAL_IDENTIFIER_SYSTEM = (
    'http://mimic.mit.edu/fhir/mimic/identifier/encounter-hosp'
)

assert ORACLE_PATH.is_file(), f'Oracle not found: {ORACLE_PATH}'
assert (FHIR_WAREHOUSE / 'Encounter.parquet').is_dir(), (
    f'FHIR Encounter table not found: {FHIR_WAREHOUSE / "Encounter.parquet"}'
)

# Read-only protects the source oracle. The FHIR Parquet files are also only
# read by this notebook.
con = duckdb.connect(str(ORACLE_PATH), read_only=True)

print(f'Oracle: {ORACLE_PATH}')
print(f'FHIR warehouse: {FHIR_WAREHOUSE}')
print(f'Encounter Parquet files: {len(list((FHIR_WAREHOUSE / "Encounter.parquet").glob("*.parquet")))}')

Oracle: ~/warehouses/mimic4-demo.db
FHIR warehouse: ~/warehouses/mimic-iv-demo/delta
Encounter Parquet files: 10


## Fetch the original MIMIC timestamps

The DuckDB oracle stores `admittime` as a timezone-less MIMIC timestamp.
It is the reference wall-clock value for this investigation.

In [2]:
oracle_sql = """
SELECT
    hadm_id,
    subject_id,
    admittime AS oracle_admittime
FROM mimiciv_hosp.admissions
ORDER BY hadm_id
"""

oracle = con.sql(oracle_sql)
print(f'Original admissions: {oracle.count("*").fetchone()[0]:,}')
oracle.limit(5)

Original admissions: 275


┌──────────┬────────────┬─────────────────────┐
│ hadm_id  │ subject_id │  oracle_admittime   │
│  int32   │   int32    │      timestamp      │
├──────────┼────────────┼─────────────────────┤
│ 20044587 │   10023771 │ 2113-08-25 07:15:00 │
│ 20093566 │   10039708 │ 2143-09-26 18:24:00 │
│ 20192635 │   10037928 │ 2177-09-04 12:05:00 │
│ 20199380 │   10005909 │ 2144-10-28 23:20:00 │
│ 20214994 │   10003400 │ 2137-02-24 10:00:00 │
└──────────┴────────────┴─────────────────────┘

## Fetch the mapped FHIR timestamps

Hospital Encounters are identified by the exact MIMIC hospital identifier
system. `identifier` is an array in the Delta table, so it is unnested before
filtering.

The demo directory can contain Parquet files from more than one generation.
`union_by_name=True` tolerates schema evolution, and `DISTINCT` prevents
duplicate physical files from creating false comparison rows.

In [3]:
fhir_sql = f"""
WITH flattened AS (
    SELECT
        unnest(identifier) AS ident,
        period.start AS fhir_period_start
    FROM read_parquet(
        '{ENCOUNTER_GLOB}',
        union_by_name = true
    )
), hospital_encounters AS (
    SELECT DISTINCT
        TRY_CAST(ident.value AS INTEGER) AS hadm_id,
        fhir_period_start,
        -- Keep the FHIR wall-clock digits. Do not cast the offset-bearing
        -- string to TIMESTAMPTZ during this diagnostic comparison.
        TRY_CAST(substr(fhir_period_start, 1, 19) AS TIMESTAMP)
            AS fhir_period_walltime
    FROM flattened
    WHERE ident.system = '{HOSPITAL_IDENTIFIER_SYSTEM}'
)
SELECT *
FROM hospital_encounters
WHERE hadm_id IS NOT NULL
ORDER BY hadm_id
"""

fhir = con.sql(fhir_sql)
print(f'FHIR hospital Encounter rows after deduplication: {fhir.count("*").fetchone()[0]:,}')
fhir.limit(5)

FHIR hospital Encounter rows after deduplication: 275


┌──────────┬───────────────────────────┬──────────────────────┐
│ hadm_id  │     fhir_period_start     │ fhir_period_walltime │
│  int32   │          varchar          │      timestamp       │
├──────────┼───────────────────────────┼──────────────────────┤
│ 20044587 │ 2113-08-25T07:15:00-04:00 │ 2113-08-25 07:15:00  │
│ 20093566 │ 2143-09-26T18:24:00-04:00 │ 2143-09-26 18:24:00  │
│ 20192635 │ 2177-09-04T12:05:00-04:00 │ 2177-09-04 12:05:00  │
│ 20199380 │ 2144-10-28T23:20:00-04:00 │ 2144-10-28 23:20:00  │
│ 20214994 │ 2137-02-24T10:00:00-05:00 │ 2137-02-24 10:00:00  │
└──────────┴───────────────────────────┴──────────────────────┘

## Join and calculate the wall-clock difference

`delta_hours` compares the original MIMIC timestamp to the FHIR timestamp
after intentionally discarding only the FHIR offset representation. A value
of `1.0` means the FHIR wall-clock value is exactly one hour later than the
original MIMIC wall-clock value.

In [4]:
con.sql(f'CREATE OR REPLACE TEMP VIEW fhir_hospital_encounters AS {fhir_sql}')

comparison_sql = """
SELECT
    a.hadm_id,
    a.subject_id,
    a.oracle_admittime,
    f.fhir_period_start,
    f.fhir_period_walltime,
    epoch(f.fhir_period_walltime - a.oracle_admittime) / 3600.0
        AS delta_hours
FROM (
    SELECT hadm_id, subject_id, admittime AS oracle_admittime
    FROM mimiciv_hosp.admissions
) AS a
INNER JOIN fhir_hospital_encounters AS f USING (hadm_id)
ORDER BY abs(delta_hours) DESC, a.hadm_id
"""

con.sql(f'CREATE OR REPLACE TEMP VIEW timezone_comparison AS {comparison_sql}')
comparison = con.sql('SELECT * FROM timezone_comparison')
comparison.limit(10)

┌──────────┬────────────┬─────────────────────┬───────────────────────────┬──────────────────────┬─────────────┐
│ hadm_id  │ subject_id │  oracle_admittime   │     fhir_period_start     │ fhir_period_walltime │ delta_hours │
│  int32   │   int32    │      timestamp      │          varchar          │      timestamp       │   double    │
├──────────┼────────────┼─────────────────────┼───────────────────────────┼──────────────────────┼─────────────┤
│ 20044587 │   10023771 │ 2113-08-25 07:15:00 │ 2113-08-25T07:15:00-04:00 │ 2113-08-25 07:15:00  │         0.0 │
│ 20093566 │   10039708 │ 2143-09-26 18:24:00 │ 2143-09-26T18:24:00-04:00 │ 2143-09-26 18:24:00  │         0.0 │
│ 20192635 │   10037928 │ 2177-09-04 12:05:00 │ 2177-09-04T12:05:00-04:00 │ 2177-09-04 12:05:00  │         0.0 │
│ 20199380 │   10005909 │ 2144-10-28 23:20:00 │ 2144-10-28T23:20:00-04:00 │ 2144-10-28 23:20:00  │         0.0 │
│ 20214994 │   10003400 │ 2137-02-24 10:00:00 │ 2137-02-24T10:00:00-05:00 │ 2137-02-24 10:00:00 

In [5]:
summary = con.sql("""
SELECT
    count(*) AS joined_rows,
    count(*) FILTER (WHERE delta_hours = 0) AS exact_wall_clock_rows,
    count(*) FILTER (WHERE delta_hours = 1) AS plus_one_hour_rows,
    count(*) FILTER (WHERE delta_hours != 0) AS any_difference_rows,
    min(delta_hours) AS minimum_delta_hours,
    max(delta_hours) AS maximum_delta_hours
FROM timezone_comparison
""").fetchone()

summary_columns = [
    'joined_rows',
    'exact_wall_clock_rows',
    'plus_one_hour_rows',
    'any_difference_rows',
    'minimum_delta_hours',
    'maximum_delta_hours',
]
summary_values = dict(zip(summary_columns, summary))
display(summary_values)

if summary_values['plus_one_hour_rows']:
    display(Markdown(
        f"**Observed:** {summary_values['plus_one_hour_rows']:,} row(s) have a FHIR wall-clock time exactly one hour later than the original. This is the expected DST-gap shape: an original `02:xx` value was served as `03:xx`."
    ))
else:
    display(Markdown(
        '**No +1 hour rows were found in this warehouse subset.** The comparison is working, but this patient subset may contain no timestamp in the spring-forward gap. Try the full FHIR warehouse if available.'
    ))

{'joined_rows': 275,
 'exact_wall_clock_rows': 275,
 'plus_one_hour_rows': 0,
 'any_difference_rows': 0,
 'minimum_delta_hours': 0.0,
 'maximum_delta_hours': 0.0}

**No +1 hour rows were found in this warehouse subset.** The comparison is working, but this patient subset may contain no timestamp in the spring-forward gap. Try the full FHIR warehouse if available.

## Highlight the divergent rows

The table below highlights all non-zero wall-clock differences. The raw FHIR
string is shown with its offset, while `fhir_period_walltime` is the same
served value with the offset removed for a wall-clock comparison.

For the intrinsic issue, the useful pattern is: `oracle_admittime` at
`02:xx`, FHIR at `03:xx`, and `delta_hours = 1.0`.

In [6]:
def display_highlighted(relation, limit=100):
    columns = relation.columns
    rows = relation.limit(limit).fetchall()
    if not rows:
        display(Markdown('_No divergent rows to display._'))
        return

    header = ''.join(f'<th>{escape(str(column))}</th>' for column in columns)
    body = []
    for row in rows:
        values = dict(zip(columns, row))
        delta = values.get('delta_hours')
        color = '#fff0c2' if delta == 1.0 else '#ffd9d9'
        cells = ''.join(
            f'<td>{escape(str(values[column]))}</td>' for column in columns
        )
        body.append(f'<tr style="background-color: {color}">{cells}</tr>')

    display(HTML(
        '<table><thead><tr>' + header + '</tr></thead><tbody>'
        + ''.join(body) + '</tbody></table>'
    ))

divergences = con.sql("""
SELECT *
FROM timezone_comparison
WHERE delta_hours != 0
ORDER BY delta_hours DESC, hadm_id
""")
display_highlighted(divergences)

_No divergent rows to display._

## A concrete reading of a +1 hour row

For a row such as:

```text
Original MIMIC:  2159-03-11 02:10:00
FHIR string:    2159-03-11T03:10:00-04:00
Comparison:     03:10 - 02:10 = +1 hour
```

the one-hour difference was introduced before the concept mapping ran.
The FHIR mapping is preserving the value it was served, not creating the
original loss. The upstream generation SQL to inspect is
`~/Documents/mimic-fhir/sql/fhir_encounter.sql:65`.

This notebook does not attempt to subtract an hour. That would incorrectly
alter genuine `03:xx` timestamps, which are indistinguishable from shifted
`02:xx` timestamps after generation.